# Petrobras Financial Statement Validator

Este projeto analisa as demonstrações financeiras da Petrobras
com dados públicos da CVM, aplicando Python, SQL e auditoria
contábil para validar consistência e detectar anomalias.

## Objetivos

- Validar consistência contábil
- Detectar anomalias trimestrais
- Consultar dados com SQL
- Visualizar KPIs financeiros com Python (matplotlib)

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

## 1. Leitura e filtragem dos dados

Nesta etapa, carregamos os dados de demonstrações financeiras da CVM e filtramos apenas os registros da Petrobras.

Também restringimos a análise para os anos de 2023 e 2024, tornando o projeto mais objetivo e focado.

In [ ]:
df = pd.read_csv(
    "../data/raw/dfp_cia_aberta_DRE_con.csv",
    sep=";",
    encoding="latin1"
)

petrobras = df[df["DENOM_CIA"].str.contains("PETROBRAS", case=False, na=False)].copy()

petrobras["DT_REFER"] = pd.to_datetime(petrobras["DT_REFER"])
petrobras = petrobras[petrobras["DT_REFER"].dt.year.isin([2023, 2024])]

petrobras[["DT_REFER", "DS_CONTA", "VL_CONTA"]].head()

## 2. Seleção das contas relevantes
Selecionamos as contas-chave para análise: Receita líquida, Lucro líquido, Caixa e equivalentes, Empréstimos e Patrimônio líquido.

In [ ]:
selected_accounts = [
    "Receita líquida",
    "Lucro líquido",
    "Caixa e equivalentes",
    "Empréstimos",
    "Patrimônio líquido"
]

df_clean = petrobras[petrobras["DS_CONTA"].isin(selected_accounts)].copy()
df_clean.head()

## 3. Padronização dos dados
Renomeamos as colunas para nomes mais intuitivos.

df_clean = df_clean.rename(columns={
    "DT_REFER": "date",
    "DS_CONTA": "account",
    "VL_CONTA": "value"
})

## 4. Armazenamento em banco de dados (SQLite)
Os dados tratados são armazenados em um banco SQLite, permitindo consultas SQL para análise posterior.

In [ ]:
conn = sqlite3.connect("../data/processed/petrobras.db")
df_clean.to_sql("financial_statements", conn, if_exists="replace", index=False)
print("Dados carregados com sucesso no SQLite.")

## 5. Análise da receita com SQL
Consultamos a evolução da receita líquida ao longo do tempo, garantindo ordenação temporal correta.

In [ ]:
query = """
SELECT date, value
FROM financial_statements
WHERE account = 'Receita líquida'
ORDER BY date
"""
revenue = pd.read_sql(query, conn)
revenue.head()

## 6. Cálculo do nível de endividamento (Debt Ratio)
Debt Ratio = Empréstimos / Patrimônio Líquido — calculado via JOIN em SQL, com proteção contra divisão por zero.

In [ ]:
query = """
SELECT
    a.date,
    a.value AS debt,
    b.value AS equity,
    a.value * 1.0 / b.value AS debt_ratio
FROM financial_statements a
JOIN financial_statements b
    ON a.date = b.date
WHERE a.account = 'Empréstimos'
  AND b.account = 'Patrimônio líquido'
  AND b.value != 0
"""
debt_ratio = pd.read_sql(query, conn)

avg_debt = debt_ratio["debt_ratio"].mean()
print(f"Debt ratio médio: {avg_debt:.2f}")
debt_ratio.head()

## 7. Visualizações principais
Analisamos a evolução da receita líquida e do debt ratio ao longo do tempo.

In [ ]:
ax = revenue.plot(
    x="date",
    y="value",
    figsize=(12, 5),
    marker="o",
    grid=True,
    title="Evolução da Receita Líquida - Petrobras"
)
ax.set_xlabel("Período")
ax.set_ylabel("Valor (R$ milhões)")  # ← unidade explícita
plt.tight_layout()
plt.show()

ax = debt_ratio.plot(
    x="date",
    y="debt_ratio",
    figsize=(12, 5),
    marker="o",
    grid=True,
    title="Debt Ratio ao longo do tempo"
)
ax.set_xlabel("Período")
ax.set_ylabel("Debt Ratio (Empréstimos / PL)")
plt.tight_layout()
plt.show()

## 8. Cálculo de variação trimestral (QoQ)
Calculamos a variação percentual entre períodos consecutivos, garantindo ordenação temporal antes do cálculo.

In [ ]:
# Garante ordenação antes de calcular variação — pct_change depende da ordem
revenue = revenue.sort_values("date").reset_index(drop=True).copy()
revenue["qoq"] = revenue["value"].pct_change()

max_growth = revenue["qoq"].max()
print(f"Maior crescimento trimestral: {max_growth:.2%}")

## 9. Identificação de anomalias
Variações superiores a 25% entre períodos consecutivos são sinalizadas como anomalias para investigação.

In [ ]:
ANOMALY_THRESHOLD = 0.25  # constante nomeada — fácil de ajustar

alerts = revenue[revenue["qoq"].abs() > ANOMALY_THRESHOLD].copy()
print(f"{len(alerts)} anomalia(s) identificada(s) (variação > {ANOMALY_THRESHOLD:.0%})")
alerts.style.format({"qoq": "{:.2%}"})

## 10. Detecção estatística de outliers (Z-score)
Aplicamos o Z-score para identificar valores estatisticamente distantes da média. Pontos além de ±2 desvios merecem investigação.

In [ ]:
ZSCORE_THRESHOLD = 2.0  # constante nomeada

revenue["zscore"] = (
    revenue["value"] - revenue["value"].mean()
) / revenue["value"].std()

ax = revenue.plot(
    x="date",
    y="zscore",
    figsize=(12, 5),
    marker="o",
    grid=True,
    title="Z-score da Receita Líquida"
)
ax.axhline(ZSCORE_THRESHOLD, linestyle="--", color="red", label=f"+{ZSCORE_THRESHOLD}")
ax.axhline(-ZSCORE_THRESHOLD, linestyle="--", color="red", label=f"-{ZSCORE_THRESHOLD}")
ax.set_xlabel("Período")
ax.set_ylabel("Z-score")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Exportação dos dados processados

Exportamos os dados tratados para CSV como camada de persistência,
permitindo reuso em análises futuras sem necessidade de reprocessar
os dados brutos da CVM.

In [ ]:
df_clean.to_csv("../data/processed/final_data.csv", index=False)
revenue.to_csv("../data/processed/revenue_analysis.csv", index=False)
alerts.to_csv("../data/processed/alerts.csv", index=False)
print("Arquivos exportados com sucesso.")

## 12. Conclusão

A análise validou a consistência dos dados financeiros da Petrobras
e identificou variações relevantes nos principais indicadores.

O uso combinado de Python, SQL e análise exploratória replicou
processos comuns em auditoria e análise financeira, com visualizações
produzidas inteiramente em Python.

Pipeline concluído:
dados brutos → tratamento → SQL → análise → alertas → visualização